Damped Newton method 
chi=30, truncating to 14,14 per leg
Start Newton method after 4 RG steps from T=T_c.
Damping with newton_step=0.5 is activated a couple of times (e.g. for i=3)
Here I ran up to i=9, reaching fp_error =3.7034621469049967e-5. 10 eigenvalues are used. 
gilt_eps=2e-5

In [1]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
gilt_eps = 2e-5 # instead of 6e-6 in the paper
chi = 30
trunc_shape = [14 14; 14 14; 14 14; 14 14]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 1,
	"rotate" => true
)
Jratio = 1.0

relT=1.0
rg_steps = 10
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.03361407516010624 and became 0.0. Index CartesianIndex(1, 16, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -0.012345916839578114 and became -1.8524388175538877e-9. Index CartesianIndex(15, 2, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466


1 

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.004606248308129038 and became 0.0. Index CartesianIndex(17, 3, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.0022689612252432567 and became 0.0. Index CartesianIndex(15, 2, 19, 2) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.0004254458368950628 and became 0.0. Index CartesianIndex(15, 21, 3, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -9.030177255728281e-5 and became 0.0. Index CartesianIndex(15, 1, 9, 16) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 6.0392199719057

[1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [2 2; 2 2; 2 2; 2 2][0 1; 0 1; 0 1; 0 1]
3 [8 8; 8 8; 8 8; 8 8][0 1; 0 1; 0 1; 0 1]
4 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
5 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
6 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
7 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
8 [14 16; 14 16; 14 16; 14 16][0 1; 0 1; 0 1; 0 1]
9 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
10 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
11 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]


In [ ]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = truncate_blocks(traj[4], trunc_shape)
for i in 1:30
    println("i=",i)
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    e0, RAshape = fp_error_with_shape(A[i],accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RAshape)
    flush(stdout)
    if A[i].shape != RAshape
        throw(ErrorException("shapes unequal"))
    end
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 10, accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    newton_step = 1.0
    enew = e0
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * deltaA[i]
        Anew, accepted_elements_new = fix_discrete_gauge(Anew; tol = 1e-7);
        enew, RAnewshape = fp_error_with_shape(Anew, accepted_elements_new, gilt_pars; trunc_shape = trunc_shape)
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnewshape)
        if enew < e0 && Anew.shape == RAnewshape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
end

i=1
||R(A[i])-A[i]||= 0.0578969047685301
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
Dict{Any, Any}((1, "N") => 28, (1, "W") => 22, (1, "S") => 25, (1, "E") => 22, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  10 eigenvalues converged
│ *  norm of residuals = (3.472417297097275e-47, 7.670419468994307e-33, 3.32619051791485e-32, 1.6922292775214578e-23, 1.7740758008179597e-23, 3.4525470502477595e-19, 3.4525470502477595e-19, 1.6077429831729145e-14, 2.16156248491095e-13, 2.16156248491095e-13)
└ *  number of operations = 51


EIGENVALUES (INITIAL):
1.9850560719513763 + 0.0im
-0.9350261455530193 + 0.0im
-0.9250188975000475 + 0.0im
0.5572873687198915 + 0.0im
0.5512743095919949 + 0.0im
0.0006860577184307256 + 0.4193743841586345im
0.0006860577184307256 - 0.4193743841586345im
-0.31065773288182197 + 0.0im
-4.96938432879862e-6 + 0.3092368920025112im
-4.96938432879862e-6 - 0.3092368920025112im
||deltaA[i]||= 0.13408277317668268
newton_step= 1.0
fp_error= 0.027492197500780795
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=2
||R(A[i])-A[i]||= 0.027492197500780795
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
Dict{Any, Any}((1, "N") => 71, (1, "W") => 46, (1, "S") => 52, (1, "E") => 45, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  13 eigenvalues converged
│ *  norm of residuals = (2.57508526750391e-54, 2.1710211080294736e-39, 4.1804278006366545e-39, 5.944514661640464e-31, 9.113239157237127e-29, 1.1717465633278477e-25, 1.1717465633278477e-25, 1.3608312570013974e-20, 1.0355392901630777e-16, 1.0134291427892338e-14, 1.0134291427892338e-14, 2.69974960092526e-15, 2.69974960092526e-15)
└ *  number of operations = 60


EIGENVALUES (INITIAL):
1.98907367784336 + 0.0im
-0.9723943895433823 + 0.0im
-0.9614565374085755 + 0.0im
0.6438170883538767 + 0.0im
0.5917392539591635 + 0.0im
-0.008371976160673741 + 0.49300654015864065im
-0.008371976160673741 - 0.49300654015864065im
-0.39600065631442516 + 0.0im
0.3455758981359368 + 0.0im
0.0002896396349431524 + 0.3072147988493002im
0.0002896396349431524 - 0.3072147988493002im
-0.12276974884170103 + 0.260960035119185im
-0.12276974884170103 - 0.260960035119185im
||deltaA[i]||= 0.07670665921015427
newton_step= 1.0
fp_error= 0.019335844644769597
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=3
||R(A[i])-A[i]||= 0.019335844644769597
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
Dict{Any, Any}((1, "N") => 95, (1, "W") => 91, (1, "S") => 66, (1, "E") => 88, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


...

EIGENVALUES (INITIAL):
1.9983338497202194 + 0.0im
-0.994446419807391 + 0.0im
-0.9866343204884975 + 0.0im
0.6128757996337133 + 0.0im
0.6012126710429032 + 0.0im
-0.008324802727561056 + 0.5034600788317961im
-0.008324802727561056 - 0.5034600788317961im
0.3850403490815062 + 0.0im
-0.3507191426872626 + 0.0im
-0.14837033338622913 + 0.2463650885734532im
-0.14837033338622913 - 0.2463650885734532im
||deltaA[i]||= 8.872711369733535e-5
newton_step= 1.0
fp_error= 9.993733523457927e-6
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=11
||R(A[i])-A[i]||= 9.993733523457927e-6
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
Dict{Any, Any}((1, "N") => 71, (1, "W") => 48, (1, "S") => 43, (1, "E") => 48, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (4.864858774831991e-53, 1.4225493765712295e-37, 2.60178414440406e-38, 6.657127460503707e-30, 2.7110545800053274e-28, 6.187404479568181e-26, 6.187404479568181e-26, 4.663869454418129e-18, 6.94685165913203e-17, 3.1688504934061284e-15, 3.1688504934061284e-15)
└ *  number of operations = 59


EIGENVALUES (INITIAL):
1.998358695455336 + 0.0im
-0.9945840926809009 + 0.0im
-0.9866491785459179 + 0.0im
0.6371304722735827 + 0.0im
0.5971590932986145 + 0.0im
-0.009686323747533794 + 0.5023680542905196im
-0.009686323747533794 - 0.5023680542905196im
0.3769884319013576 + 0.0im
-0.34832381768005555 + 0.0im
-0.14851095008416815 + 0.24567930379092393im
-0.14851095008416815 - 0.24567930379092393im
||deltaA[i]||= 2.4720562688911077e-5
newton_step= 1.0
fp_error= 1.388865081807116e-6
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=12
||R(A[i])-A[i]||= 1.388865081807116e-6
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
Dict{Any, Any}((1, "N") => 71, (1, "W") => 48, (1, "S") => 43, (1, "E") => 48, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (2.4887517808262666e-53, 1.2093807514555527e-38, 1.483691501725849e-38, 3.470558139878628e-30, 1.1485058278721669e-27, 1.7083872348343367e-26, 1.7083872348343367e-26, 2.057595427307784e-18, 2.0657091717255916e-18, 2.086436502999855e-15, 2.086436502999855e-15)
└ *  number of operations = 59


EIGENVALUES (INITIAL):
1.9983686400698 + 0.0im
-0.9946217348129841 + 0.0im
-0.9866580637273942 + 0.0im
0.6430329439041319 + 0.0im
0.5966893891091841 + 0.0im
-0.010077700054715385 + 0.5021531755847011im
-0.010077700054715385 - 0.5021531755847011im
0.37463064443948363 + 0.0im
-0.3477220556161523 + 0.0im
-0.14850628693821433 + 0.24547670700802682im
-0.14850628693821433 - 0.24547670700802682im
||deltaA[i]||= 2.203160327527167e-6
newton_step= 1.0
fp_error= 6.523289397354374e-7
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=13
||R(A[i])-A[i]||= 6.523289397354374e-7
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
Dict{Any, Any}((1, "N") => 71, (1, "W") => 48, (1, "S") => 43, (1, "E") => 48, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)
